# 04 · Polars e DuckDB: tabelas maiores, consultas mais rápidas

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

O pandas resolve bem tabelas que cabem com folga na memória. Quando o arquivo passa de alguns gigabytes, ou quando a mesma análise precisa rodar centenas de vezes, duas ferramentas mais novas fazem diferença: o **Polars**, uma biblioteca de DataFrames escrita para usar todos os núcleos do processador, e o **DuckDB**, um banco de dados analítico que roda dentro do seu notebook e consulta arquivos diretamente com SQL.

## O que você será capaz de fazer ao final

1. Dizer quando o pandas deixa de ser a melhor escolha.
2. Selecionar, filtrar, criar colunas e agrupar com as expressões do Polars.
3. Explicar a execução preguiçosa e ler um plano de consulta.
4. Consultar arquivos e DataFrames com SQL no DuckDB, incluindo funções de janela.
5. Escrever a mesma análise em pandas, Polars e SQL, e medir a diferença.
6. Passar dados de uma ferramenta para outra e para o scikit-learn.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import duckdb
import numpy as np
import pandas as pd
import polars as pl

print("polars", pl.__version__, "· duckdb", duckdb.__version__, "· pandas", pd.__version__)
assert int(pl.__version__.split(".")[0]) >= 1, "Atualize o Polars: %pip install -q -U polars  (e reinicie a sessão)"
assert int(duckdb.__version__.split(".")[0]) >= 1, "Atualize o DuckDB: %pip install -q -U duckdb  (e reinicie a sessão)"

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

## 1. Onde o pandas aperta

Três características do pandas viram limite quando os dados crescem:

| Característica | Consequência |
|---|---|
| Executa **cada operação na hora** em que é escrita | não consegue reorganizar a consulta: lê todas as colunas mesmo que você use duas; filtra depois de carregar tudo |
| Usa **um único núcleo** do processador na maior parte das operações | os outros núcleos ficam parados |
| Mantém **tudo na memória**, muitas vezes com cópias intermediárias | costuma precisar de várias vezes o tamanho do arquivo |

Polars e DuckDB atacam os três pontos: recebem a **descrição completa** da consulta antes de executar, otimizam o plano, usam todos os núcleos e leem do disco apenas o necessário.

| | pandas | Polars | DuckDB |
|---|---|---|---|
| Interface | métodos sobre DataFrames | **expressões** sobre DataFrames | **SQL** |
| Execução | imediata | imediata ou preguiçosa | preguiçosa |
| Paralelismo | pouco | sim | sim |
| Onde brilha | tabelas pequenas e médias; ecossistema enorme | transformações longas em tabelas grandes | consultas analíticas sobre arquivos; junções grandes |

## 2. Polars: primeiros passos

In [ ]:
dados = pl.read_csv(ARQUIVO, schema_overrides={"product_id": pl.String}, try_parse_dates=True)

print(dados.shape)
dados.head(3)

Três diferenças em relação ao pandas aparecem logo:

1. **Não existe índice.** As linhas não têm rótulo; são identificadas pela posição. Some com isso toda uma classe de surpresas (alinhamento automático, `reset_index`).
2. **Os tipos aparecem sob o nome de cada coluna** e são rígidos: `str`, `i64`, `f64`, `datetime`.
3. **O valor ausente é `null`**, igual para todos os tipos, e é diferente de `NaN` (que no Polars é só um número decimal inválido).

In [ ]:
dados.schema

### Os quatro verbos

Quase tudo no Polars se faz com quatro métodos, que recebem **expressões**:

| Verbo | Faz | Equivalente no pandas |
|---|---|---|
| `select` | escolhe ou calcula colunas | `df[[...]]` |
| `filter` | escolhe linhas | `df[condição]` ou `query` |
| `with_columns` | acrescenta ou substitui colunas | `assign` |
| `group_by(...).agg(...)` | resume por grupo | `groupby(...).agg(...)` |

Uma **expressão** é uma receita, construída a partir de `pl.col("nome")`, que diz o que calcular. Ela só é executada quando entregue a um desses verbos.

In [ ]:
receita = pl.col("overall_rating").mean().round(2).alias("nota_media")      # nada foi calculado ainda
print(type(receita).__name__, "→", receita)

dados.select(receita)                                                      # agora sim

In [ ]:
(
    dados
    .filter((pl.col("overall_rating") <= 2) & (pl.col("reviewer_state") == "PE"))     # & | ~ e parênteses, como no pandas
    .select("submission_date", "product_name", "overall_rating")
    .sort("submission_date", descending=True)
    .head(3)
)

Repare que não há atribuição encadeada nem dúvida entre visão e cópia: cada verbo devolve um DataFrame novo, e os dados em si nunca são alterados no lugar.

### ✏️ Exercício 1 (★): seleção em Polars

Crie `reclamacoes`: as avaliações da categoria `"Celulares e Smartphones"` com nota 1, apenas com as colunas `product_name`, `review_title` e `reviewer_state`, ordenadas pelo estado.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
reclamacoes = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert isinstance(reclamacoes, pl.DataFrame), "o resultado deve ser um DataFrame do Polars"
assert reclamacoes.columns == ["product_name", "review_title", "reviewer_state"] and 3000 < reclamacoes.height < 4000
assert reclamacoes["reviewer_state"].drop_nulls().is_sorted(), "ordene pelo estado"
print("Exercício 1 correto:", reclamacoes.shape)

## 3. Expressões

### Várias colunas de uma vez

`with_columns` recebe várias expressões e as calcula **em paralelo**. O nome da coluna nova vem do argumento nomeado.

In [ ]:
(
    dados
    .with_columns(
        nota_dobrada=pl.col("overall_rating") * 2,
        estado_minusculo=pl.col("reviewer_state").str.to_lowercase(),
        mes=pl.col("submission_date").dt.month(),
        caracteres=pl.col("review_text").str.len_chars(),
    )
    .select("overall_rating", "nota_dobrada", "reviewer_state", "estado_minusculo", "mes", "caracteres")
    .head(3)
)

Como no pandas, métodos de texto ficam sob `.str` e métodos de data, sob `.dt`.

> ⚠️ Dentro de um mesmo `with_columns`, uma expressão **não enxerga** as colunas criadas pelas outras, porque todas são calculadas ao mesmo tempo. Para usar uma coluna recém-criada, encadeie um segundo `with_columns`.

### Condições: `when`, `then`, `otherwise`

In [ ]:
idade = 2018 - pl.col("reviewer_birth_year")           # uma expressão pode ser guardada e reutilizada

(
    dados
    .with_columns(
        idade=pl.when(idade.is_between(14, 100)).then(idade),                 # sem otherwise: o resto fica null
        sentimento=pl.when(pl.col("overall_rating") <= 2).then(pl.lit("negativa"))
                     .when(pl.col("overall_rating") == 3).then(pl.lit("neutra"))
                     .otherwise(pl.lit("positiva")),
    )
    .select("reviewer_birth_year", "idade", "overall_rating", "sentimento")
    .head(4)
)

`pl.lit("texto")` marca um valor literal; sem ele, o Polars entenderia `"negativa"` como o nome de uma coluna.

### Valores ausentes

In [ ]:
dados.null_count().select("product_brand", "reviewer_birth_year", "reviewer_state", "review_text")

In [ ]:
dados.select(
    sem_estado=pl.col("reviewer_state").is_null().mean(),
    estado_preenchido=pl.col("reviewer_state").fill_null("??").n_unique(),
    media_ano=pl.col("reviewer_birth_year").mean(),               # agregações ignoram null
)

### Juntando tudo: a tabela preparada

Com o que vimos, a preparação que fizemos em pandas na aula 02 cabe em uma única cadeia. Esta é a tabela usada no restante do notebook.

In [ ]:
idade = 2018 - pl.col("reviewer_birth_year")
avaliacoes = (
    dados
    .filter(pl.col("recommend_to_a_friend").is_not_null())                       # 18 linhas sem resposta
    .with_columns(
        nao_recomenda=(pl.col("recommend_to_a_friend") == "No").cast(pl.Int8),    # comparação vira True/False; cast vira 1/0
        idade=pl.when(idade.is_between(14, 100)).then(idade),
        mes=pl.col("submission_date").dt.month(),
    )
    .drop("review_title", "review_text", "product_name")
)
avaliacoes_pd = avaliacoes.to_pandas()          # a mesma tabela em pandas, para comparações
print(avaliacoes.shape)

### ✏️ Exercício 2 (★★): uma tabela de textos

Crie `textos`, a partir de `dados`:

- apenas com as linhas em que `review_text` não é ausente;
- com `texto`: o título e o texto separados por `". "`, tratando o título ausente como texto vazio;
- com `palavras`: o número de palavras de `texto`. Como depende de `texto`, vai em um segundo `with_columns`. Dica: `pl.col("texto").str.split(" ").list.len()`;
- com `tamanho`: `"curta"` se a avaliação tem menos de 10 palavras, `"média"` se tem de 10 a 39 e `"longa"` a partir de 40;
- apenas com as colunas `reviewer_id`, `overall_rating`, `texto`, `palavras` e `tamanho`, nessa ordem.

In [ ]:
# ✏️ Exercício 2
# SEU CÓDIGO AQUI
textos = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
assert textos.columns == ["reviewer_id", "overall_rating", "texto", "palavras", "tamanho"], "confira as colunas e a ordem"
assert textos.height == dados["review_text"].drop_nulls().len(), "fique só com as linhas que têm review_text"
assert textos["texto"].null_count() == 0, "há textos ausentes: o título ausente foi tratado?"
assert set(textos["tamanho"].unique()) == {"curta", "média", "longa"}
_limites = textos.group_by("tamanho").agg(menor=pl.col("palavras").min(), maior=pl.col("palavras").max())
_limites = {linha["tamanho"]: (linha["menor"], linha["maior"]) for linha in _limites.iter_rows(named=True)}
assert _limites["curta"][1] <= 9 and _limites["média"] == (10, 39) and _limites["longa"][0] == 40, "confira os limites das três faixas"
print("Exercício 2 correto. Nota média por tamanho do texto:")
textos.group_by("tamanho").agg(avaliacoes=pl.len(), nota_media=pl.col("overall_rating").mean().round(2)).sort("nota_media", descending=True)

## 4. Agrupar

`group_by(...).agg(...)` recebe expressões, uma por coluna do resultado. `pl.len()` conta as linhas do grupo.

In [ ]:
(
    avaliacoes
    .group_by("site_category_lv1")
    .agg(
        avaliacoes=pl.len(),
        nota_media=pl.col("overall_rating").mean(),
        taxa_nao_recomenda=pl.col("nao_recomenda").mean(),
        produtos=pl.col("product_id").n_unique(),
    )
    .filter(pl.col("avaliacoes") >= 5000)                  # filtro depois da agregação (o HAVING do SQL)
    .sort("taxa_nao_recomenda", descending=True)
    .head(5)
)

> ⚠️ A ordem das linhas que saem de um `group_by` **não é garantida**, porque os grupos são processados em paralelo. Se a ordem importa, termine com `sort`.

> ⚠️ **Os ausentes formam um grupo.** No Polars (e no SQL), as linhas com `null` na coluna de agrupamento viram um grupo próprio. O pandas, por padrão, descarta essas linhas em silêncio. A mesma consulta pode dar resultados diferentes nas duas ferramentas por esse motivo; decida de forma explícita o que fazer com os ausentes.

### ✏️ Exercício 3 (★★): as piores subcategorias, em Polars

Construa `piores`: por `site_category_lv2`, as colunas `avaliacoes`, `nota_media` e `taxa_nao_recomenda`; apenas subcategorias com pelo menos 1000 avaliações; ordenada da maior para a menor taxa; as 5 primeiras. Desconsidere as avaliações sem subcategoria informada.

In [ ]:
# ✏️ Exercício 3
# SEU CÓDIGO AQUI
piores = ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_ref = (
    avaliacoes_pd.groupby("site_category_lv2")
    .agg(avaliacoes=("overall_rating", "size"), nota_media=("overall_rating", "mean"), taxa_nao_recomenda=("nao_recomenda", "mean"))
    .query("avaliacoes >= 1000").sort_values("taxa_nao_recomenda", ascending=False).head(5)
)
assert piores.columns == ["site_category_lv2", "avaliacoes", "nota_media", "taxa_nao_recomenda"] and piores.height == 5
assert piores["site_category_lv2"].null_count() == 0, "há um grupo de ausentes no resultado: retire as avaliações sem subcategoria"
assert piores["site_category_lv2"].to_list() == list(_ref.index), "as subcategorias ou a ordem diferem do resultado em pandas"
assert np.allclose(piores["taxa_nao_recomenda"].to_numpy(), _ref["taxa_nao_recomenda"].to_numpy())
print("Exercício 3 correto: mesmo resultado do pandas.")
piores

### `over`: o resumo do grupo em cada linha

No pandas usamos `transform` para devolver o resultado de um grupo a cada linha. No Polars, qualquer expressão ganha esse comportamento com `.over("coluna")`.

In [ ]:
(
    avaliacoes
    .with_columns(
        media_da_categoria=pl.col("overall_rating").mean().over("site_category_lv1"),
        avaliacoes_do_cliente=pl.len().over("reviewer_id"),
    )
    .with_columns(diferenca=pl.col("overall_rating") - pl.col("media_da_categoria"))
    .select("site_category_lv1", "overall_rating", "media_da_categoria", "diferenca", "avaliacoes_do_cliente")
    .head(4)
)

### ✏️ Exercício 4 (★★): a posição de cada avaliação entre as do mesmo produto

Crie `com_produto`, a partir de `avaliacoes`, com duas colunas novas:

- `avaliacoes_do_produto`: quantas avaliações o produto daquela linha recebeu;
- `acima_do_produto`: verdadeiro quando a nota da linha é maior que a nota média do produto.

In [ ]:
# ✏️ Exercício 4
# SEU CÓDIGO AQUI
com_produto = ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
assert com_produto.height == avaliacoes.height, "use over: o número de linhas não muda"
assert com_produto["avaliacoes_do_produto"].min() == 1 and com_produto["avaliacoes_do_produto"].max() > 500
assert com_produto["acima_do_produto"].dtype == pl.Boolean
assert not com_produto.filter(pl.col("avaliacoes_do_produto") == 1)["acima_do_produto"].any(), "produto com uma avaliação nunca supera a própria média"
print(f"Exercício 4 correto: {(com_produto['avaliacoes_do_produto'] == 1).mean():.1%} das avaliações são de produtos avaliados uma única vez.")

## 5. Execução preguiçosa

Até aqui o Polars executou cada passo imediatamente, como o pandas. O modo **preguiçoso** (*lazy*) é o que o torna realmente diferente: você descreve a consulta inteira, o Polars **otimiza o plano** e só então executa, quando você chama `collect()`.

Para entrar no modo preguiçoso, troque `read_*` por `scan_*` (ou chame `.lazy()` em um DataFrame).

In [ ]:
consulta = (
    pl.scan_csv(ARQUIVO, schema_overrides={"product_id": pl.String})        # scan: nada foi lido ainda
    .filter(pl.col("overall_rating") <= 2)
    .group_by("reviewer_state")
    .agg(negativas=pl.len())
    .sort("negativas", descending=True)
)
print(type(consulta).__name__)
print(consulta.explain())                                                   # o plano otimizado, lido de baixo para cima

Leia o plano de baixo para cima. Duas otimizações aparecem na etapa de leitura do arquivo:

- **`PROJECT 2/14 COLUMNS`**: das 14 colunas, só 2 serão lidas (o estado e a nota), porque a consulta não usa as outras. Chama-se *projection pushdown*;
- **`SELECTION`**: o filtro de nota é aplicado **durante a leitura**, e as linhas descartadas nem chegam à memória. Chama-se *predicate pushdown*.

Você escreveu "leia, filtre, agrupe"; o Polars executou "leia só o necessário, já filtrando, e agrupe". No pandas, quem precisa lembrar de passar `usecols` e de filtrar cedo é você.

In [ ]:
consulta.collect().head(3)          # só agora o arquivo é lido

### Medindo a diferença

Nossa tabela é pequena para um computador atual. Para a diferença aparecer, vamos gerar um arquivo com as avaliações repetidas 20 vezes (2,6 milhões de linhas) e fazer a mesma consulta de quatro formas.

In [ ]:
import timeit

base = avaliacoes.select("site_category_lv1", "site_category_lv2", "reviewer_state", "overall_rating", "nao_recomenda", "idade", "submission_date")
pl.concat([base] * 20).write_parquet("grande.parquet")
print(f"grande.parquet: {Path('grande.parquet').stat().st_size / 1e6:.0f} MB, " + f"{base.height * 20:,}".replace(",", ".") + " linhas")

def em_pandas():
    return pd.read_parquet("grande.parquet").query("overall_rating <= 2").groupby("site_category_lv1")["nao_recomenda"].agg(["size", "mean"])

def em_polars_imediato():
    return pl.read_parquet("grande.parquet").filter(pl.col("overall_rating") <= 2).group_by("site_category_lv1").agg(pl.len(), pl.col("nao_recomenda").mean())

def em_polars_preguicoso():
    return pl.scan_parquet("grande.parquet").filter(pl.col("overall_rating") <= 2).group_by("site_category_lv1").agg(pl.len(), pl.col("nao_recomenda").mean()).collect()

def em_duckdb():
    return duckdb.sql("SELECT site_category_lv1, COUNT(*), AVG(nao_recomenda) FROM 'grande.parquet' WHERE overall_rating <= 2 GROUP BY 1").df()

tempos = {nome: min(timeit.repeat(f, number=1, repeat=3)) for nome, f in
          [("pandas", em_pandas), ("Polars imediato", em_polars_imediato), ("Polars preguiçoso", em_polars_preguicoso), ("DuckDB", em_duckdb)]}
for nome, t in tempos.items():
    print(f"{nome:<18}{t * 1000:8.0f} ms   (velocidade em relação ao pandas: {tempos['pandas'] / t:4.1f}x)")

As versões preguiçosas (Polars com `scan` e DuckDB) ganham por uma ordem de grandeza: leem 3 das 7 colunas, filtram durante a leitura e usam todos os núcleos. O formato **Parquet** ajuda, porque guarda os dados por coluna e permite ler só as colunas pedidas.

> 📌 Os tempos variam com a máquina. No Colab gratuito, com 2 núcleos, a vantagem do paralelismo é menor do que em um computador com 8 ou 16.

Para arquivos maiores que a memória, o Polars pode processar em blocos (motor de *streaming*) e o DuckDB grava resultados intermediários em disco. As duas ferramentas permitem analisar dezenas de gigabytes em um notebook comum.

### ✏️ Exercício 5 (★★): uma consulta preguiçosa

Escreva `consulta_ex5`, uma consulta **preguiçosa** sobre `grande.parquet` que calcule, para cada `reviewer_state`, o número de avaliações (`n`) e a idade média (`idade_media`) dos clientes com nota 5, ordenada da maior para a menor `n`. Depois execute-a e guarde o DataFrame em `resultado_ex5`.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
consulta_ex5 = ...
resultado_ex5 = ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert isinstance(consulta_ex5, pl.LazyFrame), "consulta_ex5 deve ser preguiçosa: comece com pl.scan_parquet e não chame collect nela"
assert isinstance(resultado_ex5, pl.DataFrame) and resultado_ex5.columns == ["reviewer_state", "n", "idade_media"]
assert resultado_ex5["n"].is_sorted(descending=True) and resultado_ex5["reviewer_state"][0] == "SP"
assert resultado_ex5["n"].sum() == avaliacoes.filter(pl.col("overall_rating") == 5).height * 20
print("Exercício 5 correto.")
print(consulta_ex5.explain())

**Pergunta escrita.** No plano impresso acima, quantas das 7 colunas do arquivo são lidas? Por que a ordem em que você escreveu `filter` e `group_by` não afeta o desempenho no modo preguiçoso, mas afetaria no pandas?

✏️ **Sua resposta:**

_Escreva aqui._

## 6. Junções

`join` funciona como o `merge` do pandas, com os mesmos cuidados. Dois tipos extras são muito úteis: `"semi"` (mantém as linhas da esquerda que **têm** correspondência, sem trazer colunas) e `"anti"` (mantém as que **não têm**).

In [ ]:
regioes = pl.DataFrame({
    "uf": ["AC", "AP", "AM", "PA", "RO", "RR", "TO", "AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE",
           "DF", "GO", "MT", "MS", "ES", "MG", "RJ", "SP", "PR", "RS", "SC"],
    "regiao": ["Norte"] * 7 + ["Nordeste"] * 9 + ["Centro-Oeste"] * 4 + ["Sudeste"] * 4 + ["Sul"] * 3,
})

com_regiao = avaliacoes.join(regioes, left_on="reviewer_state", right_on="uf", how="left", validate="m:1")
sem_correspondencia = avaliacoes.join(regioes, left_on="reviewer_state", right_on="uf", how="anti")

print("linhas antes:", avaliacoes.height, "· depois:", com_regiao.height, "· sem região:", sem_correspondencia.height)
com_regiao.group_by("regiao").agg(n=pl.len(), taxa=pl.col("nao_recomenda").mean()).sort("taxa", descending=True)

## 7. DuckDB: SQL dentro do notebook

O **DuckDB** é um banco de dados analítico que não precisa de servidor: vem em um pacote Python e roda dentro do seu processo. A característica que o torna especial em ciência de dados é que ele consulta **arquivos e DataFrames diretamente**, sem etapa de carga.

In [ ]:
duckdb.sql("""
    SELECT site_category_lv1, COUNT(*) AS avaliacoes, ROUND(AVG(overall_rating), 2) AS nota_media
    FROM 'B2W-Reviews01.csv'                 -- o arquivo entra direto na cláusula FROM
    GROUP BY site_category_lv1
    ORDER BY avaliacoes DESC
    LIMIT 4
""").df()                                    # .df() devolve um DataFrame do pandas; .pl(), um do Polars

Os DataFrames que já existem no notebook, do pandas ou do Polars, podem ser consultados **pelo nome da variável**, sem cópia:

In [ ]:
duckdb.sql("""
    SELECT reviewer_state AS estado, COUNT(*) AS n, ROUND(AVG(nao_recomenda), 3) AS taxa
    FROM avaliacoes                          -- 'avaliacoes' é o DataFrame do Polars criado acima
    WHERE idade >= 60
    GROUP BY estado
    HAVING COUNT(*) >= 300
    ORDER BY taxa DESC
    LIMIT 5
""").pl()

### SQL em uma tabela

| Cláusula | Faz | Polars |
|---|---|---|
| `SELECT` | escolhe e calcula colunas | `select`, `with_columns` |
| `FROM` | de onde vêm os dados | o DataFrame ou `scan_*` |
| `WHERE` | filtra linhas **antes** de agrupar | `filter` antes do `group_by` |
| `GROUP BY` | define os grupos | `group_by` |
| `HAVING` | filtra grupos **depois** de agregar | `filter` depois do `agg` |
| `ORDER BY`, `LIMIT` | ordena e corta | `sort`, `head` |

Dois comandos do DuckDB ajudam muito em análise exploratória:

In [ ]:
duckdb.sql("SUMMARIZE SELECT overall_rating, idade, reviewer_state, submission_date FROM avaliacoes").df().iloc[:, :8]

### Consultas em etapas com `WITH`

Uma consulta longa fica legível quando dividida em etapas nomeadas (CTE, *common table expressions*). Cada etapa é uma tabela temporária que a seguinte pode usar.

In [ ]:
duckdb.sql("""
    WITH por_produto AS (                                   -- etapa 1: resumo por produto
        SELECT product_id, COUNT(*) AS avaliacoes, AVG(nao_recomenda) AS taxa
        FROM avaliacoes
        GROUP BY product_id
    ),
    faixas AS (                                             -- etapa 2: classifica os produtos por volume
        SELECT *, CASE WHEN avaliacoes = 1 THEN '1'
                       WHEN avaliacoes <= 10 THEN '2 a 10'
                       WHEN avaliacoes <= 100 THEN '11 a 100'
                       ELSE 'mais de 100' END AS faixa
        FROM por_produto
    )
    SELECT faixa, COUNT(*) AS produtos, SUM(avaliacoes) AS avaliacoes, ROUND(AVG(taxa), 3) AS taxa_media
    FROM faixas
    GROUP BY faixa
    ORDER BY MIN(avaliacoes)
""").df()

### Funções de janela

Uma **função de janela** calcula algo sobre um grupo de linhas e devolve o resultado **em cada linha**, como o `over` do Polars (de onde, aliás, vem o nome). A forma geral é `FUNÇÃO(...) OVER (PARTITION BY grupo ORDER BY ordem)`.

In [ ]:
duckdb.sql("""
    WITH numeradas AS (
        SELECT reviewer_id, submission_date, overall_rating,
               ROW_NUMBER() OVER (PARTITION BY reviewer_id ORDER BY submission_date) AS ordem,          -- 1ª, 2ª, 3ª avaliação do cliente
               COUNT(*)     OVER (PARTITION BY reviewer_id)                         AS total_do_cliente
        FROM avaliacoes
    )
    SELECT ordem, COUNT(*) AS avaliacoes, ROUND(AVG(overall_rating), 2) AS nota_media
    FROM numeradas
    WHERE total_do_cliente >= 3 AND ordem <= 3
    GROUP BY ordem
    ORDER BY ordem
""").df()

### ✏️ Exercício 6 (★★): as piores subcategorias, em SQL

Escreva em `sql_ex6` uma consulta que produza a mesma tabela do exercício 3: por `site_category_lv2`, as colunas `avaliacoes`, `nota_media` e `taxa_nao_recomenda`; apenas subcategorias com pelo menos 1000 avaliações; da maior para a menor taxa; 5 linhas. Desconsidere as avaliações sem subcategoria e use o DataFrame `avaliacoes` como fonte.

In [ ]:
# ✏️ Exercício 6
# SEU CÓDIGO AQUI
sql_ex6 = """
    ...
"""
resultado_ex6 = duckdb.sql(sql_ex6).pl()
resultado_ex6

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
_ref6 = (
    avaliacoes_pd.groupby("site_category_lv2")
    .agg(avaliacoes=("overall_rating", "size"), nota_media=("overall_rating", "mean"), taxa_nao_recomenda=("nao_recomenda", "mean"))
    .query("avaliacoes >= 1000").sort_values("taxa_nao_recomenda", ascending=False).head(5)
)
assert resultado_ex6.columns == ["site_category_lv2", "avaliacoes", "nota_media", "taxa_nao_recomenda"]
assert resultado_ex6["site_category_lv2"].to_list() == list(_ref6.index), "as subcategorias ou a ordem diferem do esperado"
assert np.allclose(resultado_ex6["taxa_nao_recomenda"].to_numpy(), _ref6["taxa_nao_recomenda"].to_numpy())
print("Exercício 6 correto: SQL, Polars e pandas chegam ao mesmo resultado.")

### ✏️ Exercício 7 (★★★): a pior subcategoria de cada categoria

Escreva em `sql_ex7` uma consulta que devolva, para cada `site_category_lv1`, a **subcategoria com a maior taxa de não recomendação**, considerando apenas subcategorias com pelo menos 300 avaliações. Colunas do resultado: `site_category_lv1`, `site_category_lv2`, `avaliacoes`, `taxa`. Ordene da maior para a menor taxa.

Dica: resuma por categoria e subcategoria em uma etapa `WITH`; em outra, numere as subcategorias de cada categoria com `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY taxa DESC)`; por fim, fique com as de número 1.

In [ ]:
# ✏️ Exercício 7
# SEU CÓDIGO AQUI
sql_ex7 = """
    ...
"""
resultado_ex7 = duckdb.sql(sql_ex7).pl()
resultado_ex7.head(6)

In [ ]:
# ✅ Verificação do exercício 7 (não altere)
assert resultado_ex7.columns == ["site_category_lv1", "site_category_lv2", "avaliacoes", "taxa"]
assert resultado_ex7["site_category_lv1"].n_unique() == resultado_ex7.height, "deve haver uma única linha por categoria"
assert resultado_ex7["avaliacoes"].min() >= 300 and resultado_ex7["taxa"].is_sorted(descending=True)
_maiores = avaliacoes.filter(pl.col("site_category_lv2").is_not_null()).group_by("site_category_lv1", "site_category_lv2").agg(n=pl.len(), taxa=pl.col("nao_recomenda").mean()).filter(pl.col("n") >= 300).group_by("site_category_lv1").agg(maior=pl.col("taxa").max())
_conf = resultado_ex7.join(_maiores, on="site_category_lv1")
assert np.allclose(_conf["taxa"].to_numpy(), _conf["maior"].to_numpy()), "alguma linha não é a de maior taxa da sua categoria"
print(f"Exercício 7 correto: {resultado_ex7.height} categorias.")

## 8. Três dialetos, uma pergunta

A mesma pergunta nas três ferramentas: *entre os clientes com 60 anos ou mais, quais são as três categorias com a maior taxa de não recomendação, considerando categorias com pelo menos 300 avaliações nesse grupo?*

In [ ]:
em_pandas_ = (
    avaliacoes_pd[avaliacoes_pd["idade"] >= 60]
    .groupby("site_category_lv1")
    .agg(n=("nao_recomenda", "size"), taxa=("nao_recomenda", "mean"))
    .query("n >= 300")
    .sort_values("taxa", ascending=False)
    .head(3)
    .reset_index()
)

em_polars_ = (
    avaliacoes
    .filter(pl.col("idade") >= 60)
    .group_by("site_category_lv1")
    .agg(n=pl.len(), taxa=pl.col("nao_recomenda").mean())
    .filter(pl.col("n") >= 300)
    .sort("taxa", descending=True)
    .head(3)
)

em_sql_ = duckdb.sql("""
    SELECT site_category_lv1, COUNT(*) AS n, AVG(nao_recomenda) AS taxa
    FROM avaliacoes
    WHERE idade >= 60
    GROUP BY site_category_lv1
    HAVING COUNT(*) >= 300
    ORDER BY taxa DESC
    LIMIT 3
""").pl()

print("iguais?", em_pandas_["site_category_lv1"].tolist() == em_polars_["site_category_lv1"].to_list() == em_sql_["site_category_lv1"].to_list())
em_polars_

As três versões têm a mesma estrutura: filtrar, agrupar, agregar, filtrar os grupos, ordenar e cortar. Quem entende a lógica em uma delas lê as outras duas. A escolha passa a depender do tamanho dos dados, da equipe e do restante do projeto.

## 9. Interoperabilidade

As três ferramentas trocam dados entre si com pouco ou nenhum custo, porque compartilham o formato de memória **Apache Arrow**.

In [ ]:
ida = avaliacoes.head(1000).to_pandas()            # Polars → pandas
volta = pl.from_pandas(ida)                        # pandas → Polars
via_sql = duckdb.sql("SELECT * FROM volta LIMIT 5")
print(type(ida).__name__, "|", type(volta).__name__, "|", type(via_sql.pl()).__name__, "|", type(via_sql.df()).__name__)

O **scikit-learn** aceita DataFrames do Polars diretamente, inclusive com seleção de colunas pelo nome no `ColumnTransformer`:

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

amostra = avaliacoes.sample(20_000, seed=0)
X_polars, y_polars = amostra.select("idade", "site_category_lv1"), amostra["nao_recomenda"]

modelo = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["idade"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["site_category_lv1"]),
    ]),
    LogisticRegression(max_iter=500),
)
modelo.fit(X_polars, y_polars)
print("modelo treinado a partir de um DataFrame do Polars:", type(X_polars).__name__)

Um desenho comum em projetos reais: DuckDB ou Polars fazem o trabalho pesado de leitura, filtro e junção sobre os arquivos brutos; o resultado, já reduzido, segue para o pandas ou diretamente para o scikit-learn.

## 10. Para levar

### Quando usar cada um

| Situação | Escolha razoável |
|---|---|
| Tabela de até alguns milhões de linhas, análise interativa | pandas |
| Código legado ou biblioteca que exige pandas | pandas |
| Transformação longa, repetida muitas vezes, em tabela grande | Polars preguiçoso |
| Arquivo maior que a memória | Polars com `scan_*` ou DuckDB |
| Várias junções e agregações; equipe que pensa em SQL | DuckDB |
| Consulta rápida a um arquivo Parquet ou CSV sem carregar tudo | DuckDB |

### Erros comuns

| Erro | Como evitar |
|---|---|
| Esperar índice ou alinhamento automático no Polars | não existem; use `join` e `sort` explícitos |
| Usar no mesmo `with_columns` uma coluna criada nele | encadear dois `with_columns` |
| Escrever `"texto"` onde se queria um valor literal | `pl.lit("texto")` |
| Confiar na ordem de saída do `group_by` | terminar com `sort` |
| Chamar `collect()` cedo e continuar em modo imediato | montar a consulta inteira e executar no fim |
| `read_csv` em arquivo enorme | `scan_csv` ou `scan_parquet`; converter para Parquet |
| Filtrar em `WHERE` algo que depende de agregação ou de janela | `HAVING`, ou uma etapa `WITH` adicional |

### Resumo

- pandas executa na hora, em um núcleo e na memória; Polars e DuckDB planejam, paralelizam e leem só o necessário.
- No Polars, **expressões** descrevem o cálculo e os verbos `select`, `filter`, `with_columns` e `group_by` o executam.
- Em modo **preguiçoso**, o otimizador empurra filtros e seleção de colunas para a leitura do arquivo.
- O DuckDB consulta arquivos e DataFrames com SQL, sem servidor e sem carga.
- `over` no Polars e `OVER (PARTITION BY ...)` no SQL devolvem o resumo do grupo a cada linha.
- As três ferramentas trocam dados sem custo e alimentam o scikit-learn.

### O que vem a seguir

O notebook 05 usa estes dados para treinar o primeiro modelo com o scikit-learn: pipelines, validação e os cuidados para que a avaliação seja honesta.

### Para casa

A lista de exercícios está em `04_casa.ipynb`.